# EEG i klasyfikacja wyobrażania ruchu w MNE-Python

**Pytanie badawcze:** czy na podstawie krótkiej epoki EEG można rozpoznać, czy uczestnik otrzymał instrukcję wyobrażenia ruchu obu rąk czy obu stóp? Klasyfikator **CSP + LDA** jest narzędziem do dekodowania etykiet warunku eksperymentalnego z sygnału, nie diagnostyką ani detektorem dowolnych myśli.

Pracujemy na PhysioNet EEGBCI, uczestnik 1, biegi 6/10/14. Analiza: dane → kontrola jakości → ERD/ERS → cechy CSP → LDA → walidacja po biegach. Przed uruchomieniem każdej komórki przeczytaj opis jej celu i sprawdź wynik. Dane pobiera `eegbci.load_data()` do katalogu `MNE_DATA` (domyślnie `data/`). **Rozdzielenie metod:** QC wyznaczona z całego zbioru służy tylko opisowej analizie TFR. Klasyfikator otrzymuje wszystkie epoki zdarzeń bez selekcji tym progiem. Wynik jest walidacją między biegami tej samej osoby, nie między uczestnikami.


In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
os.environ.setdefault("MNE_DATA", str(ROOT / "data"))
os.environ.setdefault("MNE_DATASETS_EEGBCI_PATH", os.environ["MNE_DATA"])

'/home/runner/work/app-eeg-mne/app-eeg-mne/mne-eeg-presentation/data'

## Slajd 06. Import bibliotek

Importujemy biblioteki potrzebne do wczytania EEG, analizy sygnału, wizualizacji i uczenia maszynowego. Uruchom komórkę i zanotuj wersje: różnice między wersjami mogą zmienić wyniki lub API.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.datasets import eegbci
from mne.io import (concatenate_raws,
                    read_raw_edf)

mne.set_log_level("WARNING")
print("MNE", mne.__version__)
print("NumPy", np.__version__)


MNE 1.13.2
NumPy 2.5.5


## Slajd 07. Pobieranie danych

Wczytamy trzy biegi **wyobrażania** ruchu u jednej osoby. Każdy bieg jest osobną jednostką walidacji. Przy pierwszym uruchomieniu pliki EDF są pobierane z PhysioNet; nie są nagraniami wykonanymi podczas tych zajęć.

In [3]:
files = eegbci.load_data(1, [6, 10, 14],
                         update_path=False)
for f in files:
    print(f.name, f.stat().st_size // 1000, "kB")


S001R06.edf 2596 kB
S001R10.edf 2596 kB
S001R14.edf 2596 kB


## Slajd 08. Wczytywanie plików EDF

Otwieramy osobno trzy zapisy EDF i zapisujemy liczbę próbek każdego z nich **przed** połączeniem. Łączymy sygnały na jednej osi czasu; granice biegów zachowamy do walidacji. Nie należy traktować ciągłości po konkatenacji jako jednej nieprzerwanej sesji.

In [4]:
raws = [read_raw_edf(f, preload=True)
        for f in files]
lens = [int(r.n_times) for r in raws]
raw = concatenate_raws(raws)
eegbci.standardize(raw)
print(lens, raw.n_times)
print(raw.ch_names[:5])


[20000, 20000, 20000] 60000
['FC5', 'FC3', 'FC1', 'FCz', 'FC2']


## Slajd 09. Metadane EEG

Odczytujemy częstotliwość próbkowania, liczbę kanałów i czas trwania. MNE przechowuje amplitudę EEG w **woltach**, ale na wykresach zwykle prezentujemy **µV**. Przykładowa wartość próbki nie oznacza poziomu aktywacji konkretnego obszaru mózgu.

In [5]:
info = raw.info
fs = info["sfreq"]
print("sfreq:", fs, "| nchan:", info["nchan"])
print("czas:", raw.times[-1] + 1 / fs, "s")
cz = raw.get_data(picks="Cz")[0]
print("Cz [V]:", cz[:3])
print("Cz [uV]:", cz[:3] * 1e6)


sfreq: 160.0 | nchan: 64
czas: 375.0 s
Cz [V]: [8.2e-05 7.7e-05 6.5e-05]
Cz [uV]: [82. 77. 65.]


## Slajd 10. Rozmieszczenie elektrod

Przypisujemy elektrodom standardowe współrzędne przestrzenne, potrzebne do map topograficznych. To **przybliżony montaż**, a nie indywidualny pomiar pozycji elektrod uczestnika.

In [6]:
raw.set_montage("colin27_1005")
pos = raw.get_montage().get_positions()
print(len(pos["ch_pos"]), "pozycji w montazu")
print("Cz [m]:", pos["ch_pos"]["Cz"].round(3))
print("C3 [m]:", pos["ch_pos"]["C3"].round(3))


64 pozycji w montazu
Cz [m]: [-0.001  0.028  0.14 ]
C3 [m]: [-0.067  0.023  0.105]


## Slajd 11. Surowy sygnał EEG

Oglądamy fragment sygnału przed filtracją. Rozpoznajemy wolne dryfy, możliwe mrugnięcia i duże wychylenia. Sam wykres nie pozwala jednoznacznie ustalić fizjologicznego źródła każdego zakłócenia.

In [7]:
seg = raw.copy().crop(20, 30)
chs = ["Fp1", "C3", "Cz", "C4", "Oz"]
data = seg.get_data(picks=chs, units="uV")
print(data.shape, np.ptp(data[1]).round(1))


(5, 1601) 419.0


## Slajd 12. Znaczniki eksperymentalne

Zliczamy adnotacje T0, T1 i T2. **T1 = obie ręce, T2 = obie stopy** tylko dla użytych biegów 6, 10 i 14. Są to etykiety zadania, nie bezpośredni pomiar tego, co badany rzeczywiście sobie wyobrażał.

In [8]:
ann = raw.annotations
desc = [d for d in ann.description
        if d[0] == "T"]
cnt = [desc.count(k) for k in ("T0","T1","T2")]
print("T0/T1/T2:", *cnt)


T0/T1/T2: 45 21 24


## Slajd 13. Widmo mocy EEG

Obliczamy widmową gęstość mocy metodą Welcha. Sprawdzamy rytmy w zakresie 8–13 Hz; szczyt widma nie dowodzi sam w sobie aktywności wyłącznie kory sensomotorycznej. Wartości PSD mają jednostkę µV²/Hz po przeskalowaniu z V²/Hz.

In [9]:
spec = raw.compute_psd(
    method="welch", fmax=45, n_fft=640)
psd = spec.get_data(picks="C3")[0]
mu = (spec.freqs >= 8) & (spec.freqs <= 13)
print("C3 mu:", (psd[mu] * 1e12).mean().round(1))


C3 mu: 43.5


## Slajd 14. Filtracja EEG

Przygotowujemy **dwie niezależne kopie** EEG: 1–40 Hz do analizy czasowo-częstotliwościowej i oceny artefaktów, 7–30 Hz do dekodowania wyobrażania ruchu. Filtr ogranicza pasmo, ale nie usuwa automatycznie mrugnięć ani ruchów mięśni.

In [10]:
raw_tfr = raw.copy().filter(
    1.0, 40.0, fir_design="firwin")
raw_clf = raw.copy().filter(
    7.0, 30.0, fir_design="firwin")
print(raw_clf.info["lowpass"], "Hz")


30.0 Hz


## Slajd 15. Referencjonowanie EEG

Stosujemy średnią referencję do obu kopii. EEG mierzy **różnice potencjałów**, więc zmiana referencji wpływa na amplitudy i mapy. Przy 64 kanałach średnia referencja wprowadza liniową zależność, istotną przy estymacji kowariancji.

In [11]:
for r in (raw_tfr, raw_clf):
    r.set_eeg_reference("average",
                        projection=False)
x = raw_clf.get_data()
print("suma = 0:", abs(x.sum(0)).max() < 1e-12)


suma = 0: True


## Slajd 16. Wykrywanie zdarzeń

Zamieniamy adnotacje na próbki zdarzeń i jawnie przypisujemy etykiety 1 oraz 2. Pomijamy T0. Sprawdź liczbę prób każdej klasy przed dalszą analizą.

In [12]:
events, _ = mne.events_from_annotations(
    raw_tfr, event_id={"T1": 1, "T2": 2})
print(events.shape)
print(events[:4])
print(np.bincount(events[:, 2])[1:])


(45, 3)
[[ 672    0    2]
 [2000    0    1]
 [3328    0    1]
 [4656    0    2]]
[21 24]


## Slajd 17. Segmentacja EEG

Wycinamy epoki EEG względem wskazówki zadania. Okno −1,5 do 4,5 s obejmuje także okres poprzedzający zdarzenie potrzebny do normalizacji mocy; samo wycięcie prób **nie** oznacza jeszcze odrzucenia artefaktów.

In [13]:
epochs = mne.Epochs(
    raw_tfr, events, dict(hands=1, feet=2),
    tmin=-1.5, tmax=4.5, baseline=None,
    reject=None, preload=True)
print(epochs.get_data().shape)


(45, 64, 961)


## Slajd 18. Kontrola jakości

**Cel:** pokazujemy detekcję skrajnych epok w eksploracyjnej analizie ERD/ERS. **Co robimy:** liczymy max peak-to-peak w paśmie 1–40 Hz, następnie próg mediany + 3 MAD. **Ograniczenie:** próg wyznaczono z całego zbioru, więc **nie** wolno użyć nim przefiltrowanych epok do raportowania niezależnej jakości klasyfikatora. W późniejszej części ML analizujemy osobno pełny zbiór zdarzeń z pasma 7–30 Hz.

In [14]:
import scipy.stats as st
pp = np.ptp(epochs.get_data(), axis=2).max(1)
mad = st.median_abs_deviation(pp, scale="normal")
thr = np.median(pp) + 3 * mad
print("200 uV:", (pp > 200e-6).sum(), "z", len(pp))
epochs.drop_bad(reject=dict(eeg=thr))
print(round(thr * 1e6), "uV;", len(epochs), "epok")


200 uV: 43 z 45
722 uV; 44 epok


## Slajd 19. Independent Component Analysis

ICA próbuje znaleźć statystycznie niezależne składowe mieszaniny. Kanał Fp1 jest tutaj **zastępnikiem**, a nie prawdziwym EOG; wykryta korelacja jedynie sugeruje potencjalny artefakt oczny. Bez niezależnej weryfikacji nie usuwamy komponentu i nie stosujemy ICA do danych klasyfikatora.

In [15]:
ica = mne.preprocessing.ICA(n_components=20,
                            random_state=97)
ica.fit(raw_tfr)
idx, sc = ica.find_bads_eog(raw_tfr, ch_name="Fp1")
print("kandydaci:", list(map(int, idx)))


kandydaci: [0]


## Slajd 20. Uśrednianie EEG

Uśredniamy epoki osobno dla warunków. Wynik `Evoked` uwydatnia odpowiedzi o podobnej fazie względem zdarzenia. Zmiany mocy rytmu μ są często niesynchronizowane fazowo, więc brak dużej różnicy w średnich potencjałach nie oznacza braku informacji w EEG.

In [16]:
evokeds = {c: epochs[c].average()
           for c in ["hands", "feet"]}
print([e.nave for e in evokeds.values()])


[21, 23]


## Slajd 21. Analiza czasowo-częstotliwościowa

Obliczamy rozkład mocy w czasie i częstotliwości przy użyciu falek Morleta. Tutaj oglądamy **moc**, a nie jeszcze procentową zmianę względem okresu przed zadaniem; nie utożsamiamy wykresu TFR z gotowym pomiarem ERD.

In [17]:
freqs = np.arange(6, 36, 1.0)
tfr = {c: epochs[c].compute_tfr(
           "morlet", freqs, n_cycles=freqs / 2,
           decim=2, average=True)
       for c in ["hands", "feet"]}
print(tfr["hands"].data.shape)


(64, 30, 481)


## Slajd 22. Normalizacja i ERD/ERS

Normalizujemy moc do okresu −0,8...−0,2 s. Ujemna zmiana procentowa oznacza ERD (spadek mocy względem baseline), dodatnia oznacza ERS (wzrost). ERD nie oznacza ujemnej amplitudy EEG.

In [18]:
erd = {c: t.copy().apply_baseline(
           (-0.8, -0.2), mode="percent")
       for c, t in tfr.items()}
win = dict(fmin=8, fmax=13, tmin=1, tmax=3)
v = erd["hands"].get_data(picks=["C3"], **win)
print("C3, hands: %.0f %%" % (100 * v.mean()))


C3, hands: -55 %


## Slajd 23. Porównanie warunków eksperymentalnych

Porównujemy zmiany mocy w kanałach C3, Cz i C4. Są to statystyki opisowe **jednej osoby**; nie dowodzą, że określony efekt wystąpi u wszystkich badanych.

In [19]:
for ch in ["C3", "Cz", "C4"]:
    h, f = [100 * erd[c].get_data(
            picks=[ch], **win).mean()
            for c in ("hands", "feet")]
    print(ch, round(h), round(f))


C3 -55 12
Cz -18 19
C4 -45 18


## Slajd 24. Analiza pasma μ

Tworzymy mapy średniej zmiany mocy w paśmie 8–13 Hz. Mapa na skórze głowy nie jest mapą aktywacji źródeł korowych: na pomiar wpływają przewodnictwo objętościowe i wybrana referencja.

In [20]:
mu = {c: 100 * erd[c].get_data(
          **win).mean((1, 2))
      for c in ["hands", "feet"]}
for c, v in mu.items():
    print(c, round(v.min()), round(v.max()))


hands -55 42
feet -16 74


## Slajd 26. Przygotowanie danych ML

**Co chcemy przewidywać?** Etykietę wskazówki wyobrażania ruchu rąk (`hands`) lub stóp (`feet`). `X` to epoka × kanał × czas, `y` to etykiety, a `groups` identyfikują trzy biegi. **Kluczowa różnica:** epoki do CSP+LDA są wycinane z **oryginalnej listy zdarzeń** (`events`), a nie z `epochs.events` po eksploracyjnym odrzucaniu. Dzięki temu dane testowe nie wpływają na wybór prób poprzez próg QC. Sprawdź liczbę epok i rozkład klas po uruchomieniu.

In [21]:
epochs_clf = mne.Epochs(
    raw_clf, events,
    dict(hands=1, feet=2),
    tmin=-1.5, tmax=4.5, baseline=None,
    reject_by_annotation=False, preload=True)
X = epochs_clf.get_data(tmin=0.5, tmax=3.5)
y = epochs_clf.events[:, 2]
t0 = epochs_clf.events[:, 0] - raw.first_samp
edges = np.cumsum(lens)
groups = np.searchsorted(edges, t0, side="right")
print(X.shape, np.bincount(y)[1:],
      np.bincount(groups))


(45, 64, 480) [21 24] [15 15 15]


## Slajd 27. Klasyfikator CSP + LDA

**Co to jest klasyfikator?** To model nadzorowany, który na podstawie przykładów z prawidłowymi etykietami uczy się przypisywać nową epokę EEG do jednej z dwóch kategorii. **Po co go budujemy?** Aby empirycznie sprawdzić, czy EEG zawiera informację umożliwiającą odróżnienie dwóch warunków wyobrażania ruchu, a nie tylko opisać średnie sygnały. **Jak działa?** CSP uczy się na danych treningowych przestrzennych kombinacji kanałów różniących się wariancją między klasami i wyznacza cztery cechy log-wariancji. LDA dopasowuje liniową granicę decyzji w przestrzeni tych cech. `Pipeline` zapewnia ponowne uczenie **CSP i LDA wewnątrz każdego foldu**. Wynik nie oznacza czytania myśli ani diagnozy.

In [22]:
from mne.decoding import CSP
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis as LDA)
from sklearn.pipeline import make_pipeline

clf = make_pipeline(
    CSP(n_components=4, reg="ledoit_wolf",
        log=True),
    LDA())
print([name for name, _ in clf.steps])


['csp', 'lineardiscriminantanalysis']


## Slajd 28. Walidacja modelu

**Jak sprawdzamy generalizację?** `GroupKFold` dopasowuje CSP oraz LDA na dwóch biegach i ocenia na trzecim, dotąd niewidzianym. Balanced accuracy to średnia czułości obu etykiet. Podział losowy to wyłącznie ćwiczenie porównawcze. **Ważne:** QC z analizy TFR nie wybiera prób ML; wynik dotyczy nowych biegów tej samej osoby, a nie innych uczestników.

In [23]:
import sklearn.model_selection as ms
cv = ms.GroupKFold(n_splits=3)
bal = dict(scoring="balanced_accuracy")
scores = ms.cross_val_score(
    clf, X, y, groups=groups, cv=cv, **bal)
skf = ms.StratifiedKFold(
    3, shuffle=True, random_state=0)
rnd = ms.cross_val_score(
    clf, X, y, cv=skf, **bal)
print("biegi :", scores.round(2))
print("losowo:", rnd.round(2))


biegi : [0.93 0.94 1.  ]
losowo: [1.   0.93 1.  ]


## Slajd 29. Ocena jakości klasyfikacji

**Jak czytać wynik?** W predykcji out-of-fold każdą epokę ocenia model, który nie trenował na tym biegu. Macierz pomyłek obrazuje rozpoznawanie każdej etykiety; balanced accuracy porównujemy z modelem większościowym (około 0,5). **Nie przenoś historycznych liczb 0,976 i 44 epok do bieżącego protokołu**: pochodziły z selekcji eksploracyjnej QC, a nowy model ocenia wszystkie epoki zadania.

In [24]:
import sklearn.metrics as sm
from sklearn.dummy import DummyClassifier
bas = sm.balanced_accuracy_score
y_hat = ms.cross_val_predict(
    clf, X, y, groups=groups, cv=cv)
ba = bas(y, y_hat)
cm = sm.confusion_matrix(y, y_hat)
dummy = DummyClassifier(
    strategy="most_frequent")
y_base = ms.cross_val_predict(
    dummy, X, y, groups=groups, cv=cv)
print(round(ba, 3), round(bas(y, y_base), 3))


0.955 0.5


## Dodatek: test permutacyjny (ok. 1,5 min)

**Po co test permutacyjny?** Porównuje jakość modelu z sytuacją, w której etykiety są losowo przemieszane w obrębie biegów. Zakładamy wymienność etykiet pod hipotezą zerową; to założenie nie wynika automatycznie z projektu doświadczenia. Przeliczaj wartość p dla bieżącego protokołu, nie kopiuj dawnego wyniku.

In [25]:
y_hat = ms.cross_val_predict(
    clf, X, y, groups=groups, cv=cv)
s, perm, p = ms.permutation_test_score(
    clf, X, y, groups=groups, cv=cv, random_state=0,
    scoring="balanced_accuracy", n_permutations=200)
print("p =", round(p, 3), "| srednia perm.:", perm.mean().round(2))


p = 0.005 | srednia perm.: 0.51


## Slajd 30. Interpretacja komponentów CSP

Dopasowanie CSP na wszystkich epokach służy wyłącznie **wizualizacji wzorców i filtrów**. Nie używamy go do obliczania wyniku walidacyjnego. Wzorce na elektrodach nie są jednoznaczną lokalizacją źródeł neuronalnych.

In [26]:
csp = CSP(n_components=4, reg="ledoit_wolf",
          log=True)
csp.fit(X, y)
print(csp.patterns_.shape, csp.filters_.shape)


(63, 64) (63, 64)


## Slajd 31. Eksport wyników

**Co zapisujemy?** Plik FIF zawiera epoki do klasyfikacji z pasma 7–30 Hz, a JSON wynik aktualnej walidacji, liczebność i wersje bibliotek. Nie utożsamiaj liczby tych epok z liczbą epok użytych w eksploracyjnej analizie TFR.

In [27]:
import json, sklearn
from pathlib import Path
out = Path("output/results")
out.mkdir(parents=True, exist_ok=True)
epochs_clf.save(out / "S001-clf-epo.fif",
                overwrite=True)
res = dict(subject=1, n_epochs=len(y),
           protocol="all_task_epochs_groupkfold_v2",
           bal_acc=round(ba, 3), mne=mne.__version__,
           sklearn=sklearn.__version__)
f = out / "results.json"
f.write_text(json.dumps(res))
print(sorted(p.name for p in out.iterdir()))


['S001-clf-epo.fif', 'results.json']


## Slajd 32. Podsumowanie i zadanie praktyczne

**Wniosek:** CSP+LDA to klasyfikator warunków zadania, nie diagnoza. Sprawdź wynik wygenerowany z bieżącego kodu dla uczestnika 1, następnie uruchom uczestnika 2. Eksperyment S1→S2 jest osobną walidacją międzyosobniczą, która wymaga uczenia wszystkich parametrów wyłącznie na S1.

In [28]:
import sys
sys.path.insert(0, "python")
from analysis import run_subject

res = run_subject(subject=1)
print(res["n_epochs"], round(res["bal_acc"], 3))
# zadanie: subject=2


45 0.955


## Ćwiczenie

1. Uruchom `run_subject(subject=2)` i porównaj walidację między biegami tej samej osoby.
2. Wskaż, dlaczego odrzucanie epok na podstawie progu wyznaczonego z wszystkich biegów przed testowaniem jest niedopuszczalne w rygorystycznej ocenie predykcyjnej.
3. Zaprojektuj trening na S1 i test na S2. Wszystkie dopasowywane parametry i transformacje muszą zostać wyznaczone bez użycia S2.
